# V3 Prompt Evaluation Results

Only two result tables are shown:

1. **Comparison** across Human A, Human B, and LLM V3.
2. **Summarized Results** showing how close LLM V3 is to the human agreement ceiling.

Problems where all three raters marked no gaps are skipped before computing the final tables.

## Setup and Data Loading

In [125]:
import json
import math
import os
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import pandas as pd

ROOT = Path('/mnt/d/Projects/kintsugi')
HUMAN_DIR = ROOT / 'dataset' / 'Rater_KC_Tags' / 'Rated_KC_V3'
LLM_DIR = ROOT / 'results' / 'human_validation' / 'llm_v3_10students'
OUTPUT_DIR = ROOT / 'results' / 'human_validation' / 'v3_prompt_eval_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STUDENT_IDS = ['10155', '9948', '14189', '14352', '14362', '14363', '14374', '14414', '14474', '14499']

KC_COLUMNS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]
VALID_KCS = set(KC_COLUMNS)
INCLUDE_EMPTY_JACCARD = False


def find_one(pattern: str, directory: Path) -> Path:
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f'No file matched {pattern} in {directory}')
    return matches[-1]


def normalize_gaps(value) -> set[str]:
    if isinstance(value, dict):
        gaps = value.get('gaps', [])
    elif isinstance(value, list):
        gaps = value
    else:
        gaps = []
    if not isinstance(gaps, list):
        return set()
    return {gap for gap in gaps if gap in VALID_KCS}


def load_annotation_file(path: Path) -> tuple[str, dict[str, set[str]]]:
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    student_id = str(data.get('studentId', data.get('student_id', 'unknown')))
    annotations = {
        f'{student_id}_{pid}': normalize_gaps(value)
        for pid, value in data.get('annotations', {}).items()
    }
    return student_id, annotations


def merge_files(file_map: dict[str, Path]) -> dict[str, set[str]]:
    merged = {}
    for expected_sid, path in file_map.items():
        loaded_sid, annotations = load_annotation_file(path)
        if loaded_sid != expected_sid:
            raise ValueError(f'Expected student {expected_sid}, found {loaded_sid} in {path.name}')
        merged.update(annotations)
    return merged


human_a_files = {sid: find_one(f'kc_annotations_Pranay Ghuge_{sid}_*.json', HUMAN_DIR) for sid in STUDENT_IDS}
human_b_files = {sid: find_one(f'kc_annotations_Arundhati Das_{sid}_*.json', HUMAN_DIR) for sid in STUDENT_IDS}
llm_files = {sid: find_one(f'llm_v3_annotations_{sid}.json', LLM_DIR) for sid in STUDENT_IDS}

human_a = merge_files(human_a_files)
human_b = merge_files(human_b_files)
llm_v3 = merge_files(llm_files)

all_common_items = sorted(
    set(human_a) & set(human_b) & set(llm_v3),
    key=lambda x: (int(x.split('_')[0]), int(x.split('_')[1]))
)

common_items = [
    item for item in all_common_items
    if human_a.get(item, set()) or human_b.get(item, set()) or llm_v3.get(item, set())
]

skipped_empty_items = len(all_common_items) - len(common_items)

print(f'Common problems before filtering: {len(all_common_items)}')
print(f'Skipped all-empty problems: {skipped_empty_items}')
print(f'Evaluated problems: {len(common_items)}')

Loaded 372 common problem annotations across 10 students.


## Metrics

In [126]:
def safe_div(num: float, den: float) -> float:
    return num / den if den else 0.0


def binary_confusion(anns_a: dict[str, set[str]], anns_b: dict[str, set[str]], items: list[str]):
    tp = fp = fn = tn = 0
    for item in items:
        gaps_a = anns_a.get(item, set())
        gaps_b = anns_b.get(item, set())
        for kc in KC_COLUMNS:
            a = kc in gaps_a
            b = kc in gaps_b
            if a and b:
                tp += 1
            elif not a and b:
                fp += 1
            elif a and not b:
                fn += 1
            else:
                tn += 1
    return tp, fp, fn, tn


def cohen_kappa(tp: int, fp: int, fn: int, tn: int) -> float:
    n = tp + fp + fn + tn
    if n == 0:
        return 0.0
    observed = (tp + tn) / n
    p_yes_a = (tp + fn) / n
    p_yes_b = (tp + fp) / n
    p_no_a = (fp + tn) / n
    p_no_b = (fn + tn) / n
    expected = (p_yes_a * p_yes_b) + (p_no_a * p_no_b)
    return safe_div(observed - expected, 1 - expected)


def gwet_ac1(tp: int, fp: int, fn: int, tn: int) -> float:
    n = tp + fp + fn + tn
    if n == 0:
        return 0.0
    observed = (tp + tn) / n
    p_yes = ((tp + fn) + (tp + fp)) / (2 * n)
    p_no = 1 - p_yes
    expected = (p_yes * p_no) + (p_no * p_yes)
    return safe_div(observed - expected, 1 - expected)


def mean_problem_jaccard(anns_a: dict[str, set[str]], anns_b: dict[str, set[str]], items: list[str]) -> tuple[float, int]:
    values = []
    for item in items:
        gaps_a = anns_a.get(item, set())
        gaps_b = anns_b.get(item, set())
        union = gaps_a | gaps_b
        if not union:
            if INCLUDE_EMPTY_JACCARD:
                values.append(1.0)
            continue
        values.append(len(gaps_a & gaps_b) / len(union))
    return (sum(values) / len(values) if values else 0.0), len(values)


def pair_metrics(name_a: str, anns_a: dict[str, set[str]], name_b: str, anns_b: dict[str, set[str]]) -> dict:
    tp, fp, fn, tn = binary_confusion(anns_a, anns_b, common_items)
    problem_jaccard, jaccard_n = mean_problem_jaccard(anns_a, anns_b, common_items)
    return {
        'Pair': f'{name_a} vs {name_b}',
        'N Problems': len(common_items),
        'N KC Decisions': len(common_items) * len(KC_COLUMNS),
        'F1 Score': safe_div(2 * tp, (2 * tp) + fp + fn),
        'Jaccard Value': problem_jaccard,
        'Jaccard N': jaccard_n,
        'Binary Jaccard': safe_div(tp, tp + fp + fn),
        'Kappa': cohen_kappa(tp, fp, fn, tn),
        'Gwet AC1': gwet_ac1(tp, fp, fn, tn),
        'Precision': safe_div(tp, tp + fp),
        'Recall': safe_div(tp, tp + fn),
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
    }


def average_metrics(rows: list[dict], pair_name: str) -> dict:
    metric_cols = ['F1 Score', 'Jaccard Value', 'Binary Jaccard', 'Kappa', 'Gwet AC1', 'Precision', 'Recall']
    averaged = {'Pair': pair_name}
    for col in metric_cols:
        averaged[col] = sum(row[col] for row in rows) / len(rows)
    averaged['N Problems'] = rows[0]['N Problems']
    averaged['N KC Decisions'] = rows[0]['N KC Decisions']
    averaged['Jaccard N'] = sum(row['Jaccard N'] for row in rows) / len(rows)
    return averaged

## Evaluation Metrics with 18 Knowledge Components (KCs)

To evaluate the agreement between Human and LLM raters across 18 specific KCs, we use a binary classification for each (1 = Gap, 0 = No Gap).

### Dataset Overview ($N=18$)

| # | KC Name | Human | LLM | Classification |
|---|---|:---:|:---:|---|
| 1 | **If/Else** | 1 | 1 | **True Positive (TP)** |
| 2 | **MathOps** | 1 | 0 | **False Negative (FN)** |
| 3 | **StringConcat** | 0 | 1 | **False Positive (FP)** |
| 4 | **LogicCompare** | 0 | 0 | True Negative (TN) |
| 5 | **ArrayIndex** | 0 | 0 | True Negative (TN) |
| 6 | **ForLoop** | 0 | 0 | True Negative (TN) |
| 7 | **WhileLoop** | 0 | 0 | True Negative (TN) |
| 8 | **FuncParams** | 0 | 0 | True Negative (TN) |
| 9 | **ReturnValue** | 0 | 0 | True Negative (TN) |
| 10 | **Recursion** | 0 | 0 | True Negative (TN) |
| 11 | **PointerUsage** | 0 | 0 | True Negative (TN) |
| 12 | **MemoryAlloc** | 0 | 0 | True Negative (TN) |
| 13 | **FileIO** | 0 | 0 | True Negative (TN) |
| 14 | **Structs** | 0 | 0 | True Negative (TN) |
| 15 | **ClassInherit** | 0 | 0 | True Negative (TN) |
| 16 | **ErrorHandling** | 0 | 0 | True Negative (TN) |
| 17 | **BooleanLogic** | 0 | 0 | True Negative (TN) |
| 18 | **TypeCasting** | 0 | 0 | True Negative (TN) |

**Summary Stats:**
* **Observed Agreement ($p_o$):** $\frac{1 + 15}{18} \approx 0.889$
* **Gaps Identified:** Human = 2, LLM = 2

---

### 1. F1 Score (Simple Case)
F1 measures the accuracy of identified gaps, ignoring the 15 KCs where both agreed no gap existed.

$$F_1 = \frac{2TP}{2TP + FP + FN}$$

**Calculation:**
$$F_1 = \frac{2(1)}{2(1) + 1 + 1} = \frac{2}{4} = \mathbf{0.500}$$

---

### 2. Cohen’s Kappa ($\kappa$)
Kappa corrects the 88.9% agreement by subtracting the agreement expected by chance.

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

* **Chance Gap ($P_{gap}$):** $\frac{2}{18} \times \frac{2}{18} \approx 0.0123$
* **Chance No Gap ($P_{none}$):** $\frac{16}{18} \times \frac{16}{18} \approx 0.7901$
* **$p_e$ (Total Chance):** $0.0123 + 0.7901 = 0.8024$

**Calculation:**
$$\kappa = \frac{0.8889 - 0.8024}{1 - 0.8024} = \frac{0.0865}{0.1976} \approx \mathbf{0.438}$$

---

### 3. Gwet’s AC1
AC1 is robust against the "prevalence paradox." In this dataset, "No Gap" is highly prevalent (83%+ of the data), which often deflates Kappa.

$$AC_1 = \frac{p_o - p_e}{1 - p_e}$$

* **Prevalence ($\pi$):** $\frac{(2/18) + (2/18)}{2} \approx 0.1111$
* **$p_e$ (Chance):** $2\pi(1 - \pi) = 2(0.1111)(0.8889) \approx 0.1975$

**Calculation:**
$$AC_1 = \frac{0.8889 - 0.1975}{1 - 0.1975} = \frac{0.6914}{0.8025} \approx \mathbf{0.862}$$

---

### Comparison Summary

| Metric | Result | Interpretation |
| :--- | :---: | :--- |
| **F1 Score** | **0.500** | Strict focus on gaps; penalizes the LLM for the single miss and single hallucination. |
| **Cohen’s Kappa** | **0.438** | Moderate agreement; heavily penalized because the high "No Gap" agreement is treated as "likely by chance." |
| **Gwet’s AC1** | **0.862** | Strong agreement; recognizes that consistently agreeing on 15/18 items is high performance despite the prevalence of zeros. |

## Comparison

In [127]:
human_ceiling = pair_metrics('Human A', human_a, 'Human B', human_b)
llm_vs_a = pair_metrics('Human A', human_a, 'LLM V3', llm_v3)
llm_vs_b = pair_metrics('Human B', human_b, 'LLM V3', llm_v3)
llm_avg = average_metrics([llm_vs_a, llm_vs_b], 'LLM V3 average vs humans')

overall_df = pd.DataFrame([human_ceiling, llm_vs_a, llm_vs_b, llm_avg])

comparison_table = overall_df[['Pair', 'N Problems', 'F1 Score', 'Jaccard Value', 'Kappa', 'Gwet AC1']].rename(columns={
    'Pair': 'Comparison',
    'N Problems': 'Problems',
    'F1 Score': 'F1',
    'Jaccard Value': 'Problem Jaccard',
})

comparison_table.style.format({
    'F1': '{:.3f}',
    'Problem Jaccard': '{:.3f}',
    'Kappa': '{:.3f}',
    'Gwet AC1': '{:.3f}',
})

,Comparison,Problems,F1,Problem Jaccard,Kappa,Gwet AC1
0,Human A vs Human B,372,0.687,0.576,0.669,0.963
1,Human A vs LLM V3,372,0.600,0.480,0.578,0.954
2,Human B vs LLM V3,372,0.558,0.435,0.536,0.952
3,LLM V3 average vs humans,372,0.579,0.458,0.557,0.953


## Summarized Results

In [128]:
summary_df = pd.DataFrame([
    {
        'Metric': metric,
        'Human Ceiling': human_ceiling[metric],
        'LLM Avg': llm_avg[metric],
        'Gap': human_ceiling[metric] - llm_avg[metric],
        '% of Ceiling': safe_div(llm_avg[metric], human_ceiling[metric]) if human_ceiling[metric] > 0 else math.nan,
    }
    for metric in ['F1 Score', 'Jaccard Value', 'Kappa', 'Gwet AC1']
])

summary_df.style.format({
    'Human Ceiling': '{:.3f}',
    'LLM Avg': '{:.3f}',
    'Gap': '{:.3f}',
    '% of Ceiling': '{:.1%}',
})

,Metric,Human Ceiling,LLM Avg,Gap,% of Ceiling
0,F1 Score,0.687,0.579,0.107,84.4%
1,Jaccard Value,0.576,0.458,0.119,79.4%
2,Kappa,0.669,0.557,0.112,83.2%
3,Gwet AC1,0.963,0.953,0.010,98.9%


## Save Results

In [129]:
ceiling_df = summary_df.rename(columns={
    'LLM Avg': 'LLM Avg vs Humans',
    'Gap': 'Gap to Ceiling',
    '% of Ceiling': 'Percent of Ceiling',
})

overall_df.to_csv(OUTPUT_DIR / 'overall_metrics.csv', index=False)
ceiling_df.to_csv(OUTPUT_DIR / 'ceiling_metrics.csv', index=False)

summary_payload = {
    'human_ceiling': human_ceiling,
    'llm_vs_human_a': llm_vs_a,
    'llm_vs_human_b': llm_vs_b,
    'llm_average_vs_humans': llm_avg,
    'ceiling': ceiling_df.to_dict(orient='records'),
    'all_common_items': len(all_common_items),
    'evaluated_items': len(common_items),
    'skipped_all_empty_items': skipped_empty_items,
    'kc_columns': KC_COLUMNS,
    'include_empty_jaccard': INCLUDE_EMPTY_JACCARD,
}

with (OUTPUT_DIR / 'v3_prompt_eval_summary.json').open('w', encoding='utf-8') as f:
    json.dump(summary_payload, f, indent=2)

print(f'Saved comparison and summary results to {OUTPUT_DIR}')

Saved comparison and summary results to /mnt/d/Projects/kintsugi/results/human_validation/v3_prompt_eval_results
